# Analysing a joint dataset

Step 3 of the ISO pipeline: an end-to-end joint analysis of the primary CMB (MFLike TT/TE/EE) and the
CMB-lensing reconstruction:

1. build the joint likelihood at a data level and evaluate it at the fiducial point;
2. fold in the **cross-covariance** from
   [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb);
3. launch an MCMC.

The configuration is an ordinary Cobaya `info` dict, so the dataset, accuracy and cosmology stay
explicit; `resolve_aliases` recovers the role aliases (`.mflike`, `.lensing`). On the **twins**
(`LEVEL = "twin"`, built by [`create_datasets.ipynb`](create_datasets.ipynb) at the fiducial we fit
at), the joint chi-square is **0 by construction**: that is the first check of the pipeline. Set
`LEVEL = "mock"` to fit the mocks themselves.

> **Configuration.** Every fiducial value, prior, modelling choice and file path comes from the
> configuration folder `$ISO_CONFIG` (by default [`example_config/`](example_config)), read through
> `iso_config.py`; the data files live under `$ISO_DATA` (by default the git-ignored `iso_data/`). See [`README.md`](README.md) for the setup. The data are used at
> three *levels*, sections of the configuration's `paths.yaml` that only differ by the files
> loaded: `mock` (realistic simulations), `twin` (their smooth twins) and `real` (the data).
>
> `iso_config.build_info(dataset, model, level=...)` builds the `info` exactly as the matching
> `cobaya-grid` job does (see [`generate_grid.ipynb`](generate_grid.ipynb)), so what is evaluated
> here is what the grid runs.

## 1. A joint likelihood

`iso_config.build_info("multigaussian_mflike_lensing", level=LEVEL)` returns the `info` of the grid
dataset of the same name: a single `MultiGaussianLikelihood` wiring MFLike + CMB lensing onto one
shared CAMB theory, reading the files of `LEVEL`. `iso_info` below fixes every parameter to the
fiducial except those in `sample`, and leaves out the cross-covariance unless asked, so the build
points below stay identical.

In [ ]:
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

import iso_config
from soliket.presets import resolve_aliases

LEVEL = "twin"  # "twin": chi^2 = 0 at the fiducial; "mock": the realistic simulations
DATASET = "multigaussian_mflike_lensing"
XCOV = iso_config.path("cross_cov", LEVEL)


def loglike(model):
    """Total log-likelihood of a model at its fiducial point."""
    return float(sum(model.loglikes({})[0]))


def iso_info(sample=(), cross_cov=False):
    """Joint MFLike + lensing info, reading the files of LEVEL.

    Every parameter is fixed to its fiducial value except those in `sample`;
    the cross-covariance is left out unless `cross_cov` is True.
    """
    info = iso_config.build_info(DATASET, level=LEVEL)
    info["params"] = iso_config.fix_to_fiducial(info["params"], sample=sample)
    if not cross_cov:
        info["likelihood"][iso_config.MULTIGAUSSIAN].pop("cross_cov_path", None)
    info["packages_path"] = resolve_packages_path()
    return info


info = iso_info()
model = get_model(info)
roles = resolve_aliases(model)

print("members :", type(roles.mflike).__name__, "+", type(roles.lensing).__name__)
print(f"joint loglike at fiducial ({LEVEL}):", round(loglike(model), 4))
if LEVEL == "twin":
    print("(expected 0: the twins are the theory at this point)")

## 2. Adding the cross-covariance

Both probes look at the same sky, so their errors are correlated. The cross-covariance is the file
produced by [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) (`cross_cov` in
`paths.yaml`) via `CrossCov.from_cmb_lensing(roles.mflike, roles.lensing)`. It keys the block by the
components' real names and carries their auto-covariances, so it drops straight into the likelihood
through `cross_cov_path`. We rebuild with `cross_cov=True` and compare the joint log-likelihood with
and without the cross term.

In [ ]:
if not XCOV.is_file():
    print("Cross-covariance not found:", XCOV)
    print("Run create_cross_covariance.ipynb (RUN_FULL = True) first to produce it.")
else:
    model_xcov = get_model(iso_info(cross_cov=True))
    print("loglike without cross-cov:", round(loglike(model), 4))
    print("loglike with    cross-cov:", round(loglike(model_xcov), 4))

## 3. Running an MCMC

Sampling is Python-native. Passing `sample=[...]` to `iso_info` keeps the named parameters sampled
(with their priors from the configuration); the `sampler` block already comes from the
configuration's `sampler.yaml`, so the `info` goes straight to `cobaya.run`. The cell below runs a
short demo chain over the amplitude `logA` (the joint fit barely constrains `tau`), guarded by
`RUN_MCMC`; a second one fits the twins made at a shifted `logA`, and both posteriors are compared. Chains are written under `$ISO_DATA/runs/<configuration>/`.

In [ ]:
RUN_MCMC = True  # set False to skip the sampler (minutes)

CHAINS = iso_config.runs_dir() / "chains" / f"multigaussian_{LEVEL}"
SAMPLED = "logA"  # the demo samples this parameter; every other one stays fixed

info_mcmc = iso_info(sample=[SAMPLED], cross_cov=XCOV.is_file())
# Short demo chain, on top of the settings of the configuration's sampler.yaml.
info_mcmc["sampler"]["mcmc"].update({"max_samples": 50, "Rminus1_stop": 0.1})

sampled = [
    p for p, v in info_mcmc["params"].items() if isinstance(v, dict) and "prior" in v
]
print("sampled parameters:", sampled)

if RUN_MCMC:
    from cobaya import run

    info_mcmc["output"] = str(CHAINS)
    updated_info, sampler = run(info_mcmc)
    sample = sampler.products()["sample"]
    print("done:", len(sample), "samples of", len(sample.sampled_params), "sampled parameters")
else:
    print("RUN_MCMC is False - skipping the sampler.")

### Recovering a shifted logA

`create_datasets.ipynb` also writes MFLike and lensing twins at `logA = 3.07` (prefix `logA3p07_`,
next to the fiducial twins). They are not in `paths.yaml`, so the cell below points both likelihoods
at those files and runs the same chain on them. The cross-covariance and every other file stay the
same: the shifted twins share them. It needs `LEVEL = "twin"`.

In [ ]:
from pathlib import Path

from cobaya import run

SHIFT_PREFIX = "logA3p07_"
SHIFTED_VALUE = 3.07  # the logA the shifted twins were made at
CHAINS_SHIFTED = iso_config.runs_dir() / "chains" / "multigaussian_twin_logA3p07"


def use_shifted_twins(info, prefix):
    """Point the MFLike and lensing likelihoods of `info` at the twins with `prefix`.

    Returns the data files now used.
    """
    file_option = {  # likelihood -> (its data-file option, the key of paths.yaml)
        "mflike.TTTEEE": ("input_file", "mflike_data"),
        "soliket.LensingLikelihood": ("data_filename", "lensing_data"),
    }
    multigaussian = info["likelihood"][iso_config.MULTIGAUSSIAN]
    files = []
    for name, options in zip(multigaussian["components"], multigaussian["options"]):
        option, key = file_option[name]
        twin = iso_config.path(key, "twin")
        options[option] = str(twin.with_name(prefix + twin.name))
        files.append(Path(options[option]))
    return files


info_shifted = iso_info(sample=[SAMPLED], cross_cov=XCOV.is_file())
info_shifted["sampler"]["mcmc"].update({"max_samples": 50, "Rminus1_stop": 0.1})
shifted_files = use_shifted_twins(info_shifted, SHIFT_PREFIX)
missing = [str(file) for file in shifted_files if not file.exists()]

if not RUN_MCMC or LEVEL != "twin":
    print("Skipping the shifted chain: it needs RUN_MCMC = True and LEVEL = 'twin'.")
elif missing:
    print("Shifted twins not found (run create_datasets.ipynb):", *missing, sep="\n  ")
else:
    info_shifted["output"] = str(CHAINS_SHIFTED)
    run(info_shifted)

In [ ]:
from getdist import loadMCSamples, plots

# The chains written above (the shifted one only if it was run), each with the logA
# value its data were made at. The first 30% of each chain is discarded as burn-in.
fiducial = iso_config.fiducial_value(info_mcmc["params"][SAMPLED])
runs = [{"root": CHAINS, "label": f"{LEVEL}, fiducial", "truth": fiducial, "color": "tab:blue"}]
if CHAINS_SHIFTED.with_name(CHAINS_SHIFTED.name + ".1.txt").exists():
    runs.append({"root": CHAINS_SHIFTED, "label": f"twin, {SAMPLED} = {SHIFTED_VALUE}",
                 "truth": SHIFTED_VALUE, "color": "tab:orange"})

samples = [loadMCSamples(str(r["root"]), settings={"ignore_rows": 0.3}) for r in runs]
for s, r in zip(samples, runs):
    print(f"{r['label']}: {s.getInlineLatex(SAMPLED, limit=1)}")  # mean and 68% interval

g = plots.get_single_plotter(width_inch=5)
g.plot_1d(samples, SAMPLED, line_args=[{"color": r["color"]} for r in runs])
ax = g.subplots[0, 0]
for r in runs:
    ax.axvline(r["truth"], color=r["color"], ls="--")  # where the data were made
g.add_legend([r["label"] for r in runs])

### A YAML twin for `cobaya-run`

The Python run above can be serialized to YAML and launched from the command line. The file holds
machine-specific absolute paths, so it is written under
`$ISO_DATA/runs/<configuration>/`, never into the tracked files:

```bash
cobaya-run <the file printed by the cell below>
```

In [ ]:
import yaml

info_yaml = iso_info(sample=[SAMPLED], cross_cov=XCOV.is_file())
info_yaml["sampler"]["mcmc"].update({"max_samples": 50, "Rminus1_stop": 0.1})
info_yaml["output"] = str(CHAINS)

header = (
    f"# ISO joint analysis ({LEVEL}) -- YAML twin of analyse_datasets.ipynb.\n"
    "# Machine-specific paths; do not commit. Run with:  cobaya-run <this file>\n\n"
)
text = yaml.safe_dump(info_yaml, sort_keys=False, default_flow_style=False)

dest = iso_config.runs_dir() / f"multigaussian_{LEVEL}.yaml"
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_text(header + text)
print("wrote", dest)

## Recap

- `iso_info()` gives the joint MFLike + CMB-lensing `info` at the fiducial, built by
  `iso_config.build_info` from the configuration and reading the files of `LEVEL`;
  `resolve_aliases(model)` recovers `.mflike` / `.lensing`.
- The cross-covariance from [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) folds in
  through `cross_cov_path`.
- Sampling (here of `logA`, on the twins at the fiducial and at a shifted `logA`) runs by calling
  `cobaya.run(info)` (the sampler comes from the configuration); the same
  config is written to `$ISO_DATA/runs/<configuration>/` for `cobaya-run`.

The same notebook runs on the real data by setting `LEVEL = "real"` once that section of
`paths.yaml` exists. The full set of runs is generated by [`generate_grid.ipynb`](generate_grid.ipynb).